# 🏀 NBA AI Referee (v3.0 Full-Stack Overhaul) — E-BARD Basketball Perception, Video Swin3D-T + Gated Attention MIL & Multi-Window RL Officiating
**Course:** CSC 434 — Artificial Intelligence & Machine Learning Final Project  
**Hardware:** Google Colab NVIDIA Tesla T4 GPU / Apple Silicon MPS / CPU (Cross-Platform)  
**Core Stack:** E-BARD NBA Basketball YOLO (`player`, `referee`, `basketball`, `hoop`), Unsupervised 2-Team CIELAB Jersey Clustering, YOLOv11-Pose, Video Swin Transformer (`Swin3D-T`, Kinetics-400), Gated Attention Multiple Instance Learning (MIL), Double Dueling DQN RL Referee

---

### 🚀 What Changed in the v3.0 Full-Stack Overhaul
1. **Zero-Referee / Zero-Spectator Basketball Perception (`2_preprocess_tubes.py`)**:
   - Replaces generic COCO `person` heuristics with **E-BARD NBA Basketball YOLO** (`GabrieleGiudici/E-BARD-detection-models`, CC-BY-4.0), natively separating **`basketball`**, **`hoop`**, active on-court **`player`**, and **`referee`**.
   - Adds a **full-length dark-trousers + gray-shirt** secondary referee guard and **Clip-Level 2-Team CIELAB K-Means Clustering** so referees and spectators are **never** boxed as players and teammates are never paired.
2. **3D Foot-Plane Depth Alignment & 16-D Rule 12 Kinematics**:
   - Modulates 2D torso overlap by foot-plane alignment (`|y2_a - y2_b|`) to eliminate broadcast depth illusions, and adds native E-BARD `hoop_proximity`, `ball_verified_on_play`, and `shooting_motion_score`.
3. **Multi-Window Video Swin3D-T (`768-D`) + Gated Attention MIL (`3_model_and_dataset.py`)**:
   - Upgrades the visual backbone from `R(2+1)D-18` (57.5% K400) to frozen **`Swin3D-T`** (78.8% K400) over **overlapping 2.0s candidate windows** covering the entire play with zero blind spots.
   - Trains a regularized **Gated Attention MIL** head (`~439K` trainable params) + real ground-truth **`call_type` head** (`Shooting Foul` vs. `Personal Foul` directly from L2M metadata—no fake pose targets!).
4. **Leak-Free Game-Grouped Validation & Honest Calibration (`4_train_eval.py`)**:
   - Splits strictly by **`game_id`** (`StratifiedGroupKFold`) + deduplicates conflicting MP4 hashes + applies post-hoc **Temperature Scaling** (zero hardcoded probability hacks or threshold clamping).
5. **Sliding-Window Any-Length Inference & Clean Conditional Video Annotation**:
   - Scans raw NBA footage of **any duration** via Temporal NMS:
     - **0 Fouls Detected**: **Nothing happens** on the output video (100% clean unaltered playback).
     - **1+ Fouls Detected**: Highlights **WHAT foul it is**, **WHERE it occurred** (opposing-team boxes, red zone, contact reticle, court zone—zero referees!), and **WHEN it occurred** strictly during each foul window.

```text
+---------------------------------------------------------------------------------------------------+
| STAGE 1: NBA L2M Scraper, Parallel Video Downloader & Content-Hash Deduplicator (Cell 2)          |
|  • Scrapes official.nba.com L2M reports (CC, INC, CNC, IC) & downloads broadcast MP4 clips        |
|  • Deduplicates conflicting/identical MP4 hashes & isolates data/test/ by stem + byte hash        |
+-------------------------------------------------+-------------------------------------------------+
                                                  |
                                                  v
+---------------------------------------------------------------------------------------------------+
| STAGE 2: E-BARD Basketball Perception + 2-Team LAB Clustering + Multi-Window Swin3D-T (Cell 4)    |
|  • E-BARD YOLO: native {basketball, hoop, player, referee} -> explicit referee/spectator removal  |
|  • Clip-Level 2-Team CIELAB K-Means -> hard opposing-team pairing + 3D foot-plane depth gating    |
|  • Slices wide-court live play into overlapping 2.0s windows -> (K, 768) Swin3D-T + (K, 16, 16)   |
+-------------------------------------------------+-------------------------------------------------+
                                                  |
                                                  v
+---------------------------------------------------------------------------------------------------+
| STAGE 3 & 4: Swin3D-T + Gated Attention MIL & Leak-Free Game-Grouped Training (Cells 5 & 6)       |
|  • Dual-Stream Window Fusion (768-D Swin3D-T + 192-D Temporal Pose Attention -> 256-D/window)     |
|  • Gated Attention MIL (ICML 2018) + Instance Neg-Suppression + Real L2M `call_type` Supervision  |
|  • 5-Fold Game-Grouped CV (`game_id`) + Post-Hoc Temperature Scaling + Honest Threshold Sweep     |
+-------------------------------------------------+-------------------------------------------------+
                                                  |
                                                  v
+---------------------------------------------------------------------------------------------------+
| STAGE 5 & 6: Multi-Window Sequential RL Referee Agent (Cells 7 & 8)                               |
|  • Distinct per-window Swin3D-T + MIL prob + 16-D kinematic trajectory (zero static tiling!)      |
|  • Double Dueling DQN trained with real L2M review rewards (CC, INC, CNC, IC)                     |
+-------------------------------------------------+-------------------------------------------------+
                                                  |
                                                  v
+---------------------------------------------------------------------------------------------------+
| STAGE 7 & 8: Drop-Folder Any-Length AI Referee Annotation & HTML5 Replay Console (Cells 9 & 10)   |
|  • Drop ANY raw NBA footage into data/test/ -> Sliding-Window Temporal NMS -> data/annotated/     |
|  • Highlights WHAT, WHERE (zero referees!), WHEN on fouls; NOTHING happens when no foul occurs!   |
+---------------------------------------------------------------------------------------------------+
```

In [ ]:
# ==============================================================================
# CELL 1: Universal Bootstrap (Colab T4 / Mac MPS / CPU), E-BARD Setup & Config
# ==============================================================================
import importlib
import os
import sys
import csv
import gc
import json
import time
import random
import platform
import subprocess
from pathlib import Path
from collections import Counter

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('🚀 Running in Google Colab environment!')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'opencv-python-headless', 'scikit-learn', 'matplotlib'], check=False)
    repo_dir = Path('/content/AI-ML-Final-Project')
    if not repo_dir.exists():
        print('📥 Cloning repository from GitHub (testing branch)...')
        subprocess.run(['git', 'clone', '-b', 'testing', 'https://github.com/ferraromike/AI-ML-Final-Project.git', str(repo_dir)], check=False)
    else:
        subprocess.run(['git', '-C', str(repo_dir), 'pull', 'origin', 'testing'], check=False)
    target_dir = repo_dir / 'NBA-Foul-Classifier-WIP'
    os.chdir(str(target_dir if target_dir.exists() else repo_dir))
    print(f'📁 Working directory set to: {Path.cwd()}')

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '2')
os.environ.setdefault('MKL_NUM_THREADS', '2')
os.environ.setdefault('YOLO_CONFIG_DIR', '/tmp/Ultralytics')

import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

cv2.setNumThreads(0)

CWD = Path.cwd().resolve()
if (CWD / 'NBA-Foul-Classifier-WIP').exists():
    PROJECT_DIR = CWD / 'NBA-Foul-Classifier-WIP'
elif (CWD / '3_model_and_dataset.py').exists():
    PROJECT_DIR = CWD
elif (CWD.parent / 'NBA-Foul-Classifier-WIP').exists():
    PROJECT_DIR = CWD.parent / 'NBA-Foul-Classifier-WIP'
else:
    PROJECT_DIR = CWD

for p in [PROJECT_DIR, PROJECT_DIR.parent]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

DATA_DIR = PROJECT_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
METADATA_CSV = DATA_DIR / 'l2m_metadata.csv'
INCORRECT_CSV = DATA_DIR / 'l2m_incorrect_fouls.csv'
PROCESSED_METADATA_CSV = PROCESSED_DIR / 'processed_metadata.csv'
WEIGHTS_DIR = PROJECT_DIR / 'weights'
TEST_DIR = DATA_DIR / 'test'
ANNOTATED_DIR = DATA_DIR / 'annotated'

CHECKPOINT_PATH = WEIGHTS_DIR / 'best_foul_r2plus1d.pt'
RL_CHECKPOINT_PATH = WEIGHTS_DIR / 'referee_rl_agent.pt'

TEST_DIR.mkdir(parents=True, exist_ok=True)
ANNOTATED_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

IS_CUDA = torch.cuda.is_available()
VRAM_GB = (torch.cuda.get_device_properties(0).total_memory / 1e9) if IS_CUDA else 0.0
IS_T4_GPU = IS_CUDA and (VRAM_GB >= 12.0)

train_mod = importlib.import_module('4_train_eval')
preprocess_mod = importlib.import_module('2_preprocess_tubes')
model_mod = importlib.import_module('3_model_and_dataset')

DEVICE = train_mod.select_compute_device()
ebard_weights = preprocess_mod.ensure_ebard_basketball_weights()

CONFIG = {
    # Stage 1: Scraping, Auto-Download (Colab/Fresh Clone) & Deduplication
    'max_games': 60,
    'max_scrape_clips': 250,
    'run_live_scrape': False,            # Set True to scrape brand-new games from official.nba.com
    'auto_download_missing_clips': True, # Auto-downloads MP4s listed in l2m_metadata.csv when data/raw/ is empty (Colab!)
    'max_download_clips': 0,             # 0 = download all clips in l2m_metadata.csv (or cap e.g. 160 for faster Colab run)

    # Stage 2: E-BARD Perception + Multi-Window Swin3D-T MIL Bags
    'max_preprocess_clips': 0,       # 0 = preprocess ALL unique real raw videos
    'max_live_play_sec': 24.0,       # Covers full ~20s live L2M play (inference scans ANY length!)
    'window_sec': 2.0,               # 2.0s candidate temporal window
    'stride_sec': 1.0,               # 1.0s sliding-window stride
    'max_bag_windows': 10,           # Up to 10 windows/clip (zero blind spots across 20s)
    'num_frames': 16,
    'crop_size': 224,
    'yolo_batch_size': 64 if IS_T4_GPU else (32 if IS_CUDA else 16),

    # Stage 4: Swin3D-T + Gated Attention MIL Training (Game-Grouped CV)
    'epochs': 25,
    'batch_size': 16,
    'lr': 4.0e-4,
    'weight_decay': 3.0e-2,
    'dropout': 0.40,
    'cv_folds': 5,                   # Leak-free 5-fold StratifiedGroupKFold by game_id

    # Stage 5: Multi-Window Sequential RL Referee
    'train_rl': True,
    'rl_episodes': 250,
    'seed': 42,
}

print(f'🖥️  Platform                : {platform.system()} ({platform.machine()})')
print(f'🔥 PyTorch Version         : {torch.__version__}')
print(f'⚡ Compute Device          : {DEVICE}' + (f' ({VRAM_GB:.1f} GB VRAM)' if IS_CUDA else ''))
print(f'🏀 E-BARD Basketball Model : {ebard_weights if ebard_weights else "Offline COCO fallback"}')
print(f'📂 Raw Footage Test Folder : {TEST_DIR} (Drop ANY raw NBA video here to test!)')
print(f'🎬 Annotated Output Folder : {ANNOTATED_DIR}')

---
## Stage 1: NBA Last Two Minute (L2M) Scraper, Downloader & Conflict Deduplication
Official NBA Last Two Minute (L2M) reports supply ground-truth officiating labels (`CC`/`INC` = Foul, `CNC`/`IC` = Legal/No Foul) plus real `call_type` (`Shooting Foul` vs. `Personal Foul`).

In v3.0, **automatic content-hash deduplication** (`deduplicate_raw_dataset`) removes any duplicate NBA Stats video assets and resolves conflicting labels (where two L2M events share the exact same `.mp4` clip) before preprocessing.

In [ ]:
# ==============================================================================
# CELL 2: Stage 1 — L2M Data Sync, Auto-Download (Colab) & Conflict Deduplication
# ==============================================================================
import shutil
scrape_mod = importlib.import_module('1_scrape_l2m')
importlib.reload(scrape_mod)

if CONFIG['run_live_scrape'] or not METADATA_CSV.exists():
    print(f"🌐 Scraping NBA L2M reports (max_games={CONFIG['max_games']}, max_clips={CONFIG['max_scrape_clips']})...")
    scrape_mod.run_training_dataset_scraper(
        output_dir=RAW_DIR,
        metadata_csv=METADATA_CSV,
        daily_csv=INCORRECT_CSV,
        include_archive=True,
        max_games=CONFIG['max_games'],
        max_clips=CONFIG['max_scrape_clips'],
    )

# Raw .mp4 videos are excluded from Git (.gitignore), so on a fresh Google Colab runtime
# `data/raw/` starts empty even though `l2m_metadata.csv` is present. Auto-download them here!
foul_clips = list((RAW_DIR / 'foul').glob('*.mp4')) if (RAW_DIR / 'foul').exists() else []
no_foul_clips = list((RAW_DIR / 'no_foul').glob('*.mp4')) if (RAW_DIR / 'no_foul').exists() else []
existing_raw_count = len(foul_clips) + len(no_foul_clips)

if CONFIG['run_live_scrape'] or existing_raw_count == 0 or (CONFIG['auto_download_missing_clips'] and existing_raw_count < 40):
    print(f"📥 Found {existing_raw_count} raw MP4(s) on disk — downloading L2M broadcast clips from NBA CDN in parallel...")
    scrape_mod.bulk_download_missing_clips(
        metadata_csv=METADATA_CSV,
        output_dir=RAW_DIR,
        max_workers=12 if IN_COLAB else 8,
        max_clips=CONFIG['max_download_clips'],
    )

# Deduplicate conflicting & identical video content hashes
dedup_stats = scrape_mod.deduplicate_raw_dataset(metadata_csv=METADATA_CSV, raw_dir=RAW_DIR)

# If data/test/ is empty (e.g., fresh Google Colab clone), automatically hold out up to 4 clips
# from 1 unseen NBA game (2 foul + 2 no_foul) BEFORE Stage 2 preprocessing & training!
existing_test_clips = train_mod.discover_video_files_in_folder(TEST_DIR)
if not existing_test_clips:
    foul_clips = sorted((RAW_DIR / 'foul').glob('*.mp4')) if (RAW_DIR / 'foul').exists() else []
    no_foul_clips = sorted((RAW_DIR / 'no_foul').glob('*.mp4')) if (RAW_DIR / 'no_foul').exists() else []
    by_game_f = {}
    by_game_nf = {}
    for p in foul_clips:
        by_game_f.setdefault(p.stem.split('_')[0], []).append(p)
    for p in no_foul_clips:
        by_game_nf.setdefault(p.stem.split('_')[0], []).append(p)
    shared_games = sorted(set(by_game_f.keys()) & set(by_game_nf.keys()))
    if len(shared_games) >= 6:
        holdout_gid = shared_games[-1]
        chosen = by_game_f[holdout_gid][:2] + by_game_nf[holdout_gid][:2]
        for p in chosen:
            shutil.move(str(p), str(TEST_DIR / p.name))
        print(f"🧪 Held out {len(chosen)} unseen clips from NBA Game {holdout_gid} into {TEST_DIR} for Cell 9/10 testing.")

foul_clips = list((RAW_DIR / 'foul').glob('*.mp4')) if (RAW_DIR / 'foul').exists() else []
no_foul_clips = list((RAW_DIR / 'no_foul').glob('*.mp4')) if (RAW_DIR / 'no_foul').exists() else []
test_clips_now = train_mod.discover_video_files_in_folder(TEST_DIR)

print("=" * 76)
print("✅ Stage 1 Data Sync, Auto-Download & Deduplication Complete!")
print(f"   • Deduplicated CSV Rows   : {dedup_stats['total_after']} (dropped {dedup_stats['dropped_conflicts']} label conflicts, {dedup_stats['dropped_dups']} duplicates)")
print(f"   • Downloaded Train MP4s   : {len(foul_clips) + len(no_foul_clips)} ({len(foul_clips)} foul, {len(no_foul_clips)} no_foul)")
print(f"   • Isolated Test MP4s      : {len(test_clips_now)} in {TEST_DIR}")
print("=" * 76)

if len(foul_clips) + len(no_foul_clips) == 0:
    raise RuntimeError(
        "No raw MP4 clips were downloaded into data/raw/! Check internet access to official.nba.com / ak-static.cms.nba.com "
        "or upload your raw NBA MP4s into data/raw/foul/ and data/raw/no_foul/."
    )

---
## Exploratory Data Analysis (EDA): Games, Review Decisions & Real `call_type` Distribution
Inspecting the deduplicated L2M dataset across unique NBA `game_id`s, binary foul balance, official review decisions (`CC`, `INC`, `CNC`, `IC`), and real `call_type` (`Shooting Foul` vs. `Personal Foul`).

In [ ]:
# ==============================================================================
# CELL 3: Dataset EDA — Game Count, Class Balance, Review Decisions & Call Types
# ==============================================================================
with open(METADATA_CSV, 'r', encoding='utf-8') as f:
    meta_rows = list(csv.DictReader(f))

unique_games = {r.get('game_id', '').strip() for r in meta_rows if r.get('game_id')}
decisions = Counter(r.get('review_decision', '').strip() for r in meta_rows if r.get('review_decision'))
labels = Counter(int(r.get('label', 0)) for r in meta_rows)
call_types = Counter(r.get('call_type', 'Personal Foul').strip() for r in meta_rows if int(r.get('label', 0)) == 1)

print(f"📊 Master Deduplicated L2M Dataset ({len(meta_rows)} plays across {len(unique_games)} unique NBA games):")
print(f"   • Fouls (Label 1)            : {labels[1]} ({labels[1]/max(len(meta_rows),1)*100:.1f}%)")
print(f"   • Legal / No Foul (Label 0)  : {labels[0]} ({labels[0]/max(len(meta_rows),1)*100:.1f}%)")
print(f"   • NBA Review Decisions       : {dict(decisions)}")
print(f"   • Positive Foul Call Types   : {dict(call_types)}")

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4.2), dpi=100)
ax1.bar(['No Foul (0)', 'Foul (1)'], [labels[0], labels[1]], color=['#2ca02c', '#d62728'], width=0.5)
ax1.set_title('Binary Ground Truth Balance', fontweight='bold')
ax1.set_ylabel('Plays')

dec_order = ['CNC', 'CC', 'INC', 'IC']
ax2.bar(dec_order, [decisions.get(d, 0) for d in dec_order], color=['#2ca02c', '#d62728', '#ff7f0e', '#9467bd'], width=0.55)
ax2.set_title('NBA L2M Review Decisions', fontweight='bold')

ct_items = call_types.most_common(4)
ax3.barh([c[0] for c in ct_items], [c[1] for c in ct_items], color='#1f77b4', height=0.5)
ax3.set_title('Real Foul Call Types (Positive Fouls)', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Stage 2: E-BARD Basketball Perception, 2-Team CIELAB Clustering & Multi-Window Swin3D-T Bag Extraction (`2_preprocess_tubes.py`)
1. **E-BARD NBA Basketball Detector (`ebard_basketball_yolo.pt`)**: Explicitly separates `basketball`, `hoop`, active on-court `player`, and `referee` — completely eliminating referee and spectator contamination.
2. **Unsupervised 2-Team CIELAB Jersey Clustering**: Fits 2-cluster K-Means across every on-court player's central torso crop in CIELAB space so only **opposing-team pairs** (`Team 0` vs. `Team 1`) are evaluated.
3. **16-D Depth-Gated Rule 12 Kinematics**: Fuses clean E-BARD players with `yolo11n-pose.pt` 17-point skeletons and gates 2D cylinder overlap by 3D foot-plane alignment (`|y2_a - y2_b|`).
4. **Multi-Window MIL Bags + Swin3D-T (`768-D`)**: Slices the full wide-court play into up to 10 overlapping `2.0s` windows and extracts Kinetics-400 **`Swin3D-T`** embeddings per window.

In [ ]:
# ==============================================================================
# CELL 4: Stage 2 — E-BARD Perception + Multi-Window Swin3D-T MIL Bag Extraction
# ==============================================================================
preprocess_mod = importlib.import_module('2_preprocess_tubes')
importlib.reload(preprocess_mod)

print("⚙️  Running Stage 2 E-BARD + 2-Team CIELAB + Swin3D-T Multi-Window Preprocessing...")
processed_rows = preprocess_mod.preprocess_dataset(
    metadata_csv=METADATA_CSV,
    processed_dir=PROCESSED_DIR,
    processed_metadata_csv=PROCESSED_METADATA_CSV,
    test_dir=TEST_DIR,
    yolo_weights=preprocess_mod.EBARD_WEIGHTS_NAME,
    pose_weights=str(WEIGHTS_DIR / 'yolo11n-pose.pt'),
    window_sec=CONFIG['window_sec'],
    stride_sec=CONFIG['stride_sec'],
    num_frames=CONFIG['num_frames'],
    crop_size=CONFIG['crop_size'],
    device='cuda' if IS_CUDA else 'auto',
    max_live_play_sec=CONFIG['max_live_play_sec'],
    max_clips=CONFIG['max_preprocess_clips'],
    yolo_batch_size=CONFIG['yolo_batch_size'],
    max_bag_windows=CONFIG['max_bag_windows'],
    synthetic_fallback=False,
)

print(f"✅ Stage 2 complete! {len(processed_rows)} clean v3 MIL bag tensors ready in {PROCESSED_DIR}")
gc.collect()
if IS_CUDA:
    torch.cuda.empty_cache()

---
## Stage 3: Video Swin3D-T + Gated Attention Multiple Instance Learning Architecture (`3_model_and_dataset.py`)
Instead of overfitting 33.8M parameters on ~140 weak clip labels from a single guessed crop, `FoulMILClassifier` keeps the Kinetics-400 **`Swin3D-T`** video backbone frozen (`768-D` per 2.0s window) and trains a compact, heavily regularized **`~439K`-parameter** Dual-Stream Gated Attention MIL network:
- **Per-Window Dual-Stream Fusion**: `768-D Swin3D-T` $\oplus$ `192-D 4-Head Temporal Pose Attention (16x16)` $\to$ `256-D` per window.
- **Per-Window Localization Heads**: `window_foul_head` (WHEN!) + `window_type_head` (WHAT — trained on **real L2M `call_type`**!).
- **Gated Attention MIL Pooling (Ilse et al., ICML 2018)**: Pushes *every* window of `no_foul` clips toward `0` and focuses attention on the exact peak foul window of `foul` clips.

In [ ]:
# ==============================================================================
# CELL 5: Inspect v3 Swin3D-T + Gated Attention MIL Model & Dataset Split
# ==============================================================================
model_mod = importlib.import_module('3_model_and_dataset')
importlib.reload(model_mod)

base_dataset = model_mod.L2MFoulDataset(
    processed_dir=PROCESSED_DIR,
    metadata_csv=METADATA_CSV,
    test_dir=TEST_DIR,
    is_train=False,
)
model_preview = model_mod.FoulMILClassifier(dropout_p=CONFIG['dropout']).to(DEVICE)
summary = model_preview.trainable_parameter_summary()

unique_games = sorted({m['game_id'] for m in base_dataset.samples_meta})
print("🏗️  FoulMILClassifier (v3.0 Swin3D-T + Gated Attention MIL) Summary:")
print(f"   • Frozen Visual Backbone : Video Swin3D-T (Kinetics-400, {summary['backbone_params_frozen']:,} params, 768-D/window)")
print(f"   • Trainable MIL + Pose   : {summary['trainable_params']:,} params")
print(f"   • Real MIL Bags Loaded   : {len(base_dataset)} clips across {len(unique_games)} unique NBA games")
print(f"   • Class Split            : {sum(m['label']==1 for m in base_dataset.samples_meta)} fouls | {sum(m['label']==0 for m in base_dataset.samples_meta)} no_foul")
print(f"   • Real Call Types (Pos)  : {Counter(m['call_type'] for m in base_dataset.samples_meta if m['label']==1)}")

---
## Stage 4: Leak-Free Game-Grouped Cross-Validation & Post-Hoc Temperature Calibration (`4_train_eval.py`)
We evaluate generalization strictly across **unseen NBA games** (`StratifiedGroupKFold` grouped by `game_id`), so no two plays from the same game, arena, or team matchup ever cross train and validation.

At the end of validation, **Temperature Scaling** calibrates honest foul probabilities and sweeps the decision threshold without hardcoded probability overrides.

In [ ]:
# ==============================================================================
# CELL 6: Leak-Free Game-Grouped 5-Fold Ensemble & Global OOF Calibration
# ==============================================================================
train_mod = importlib.import_module('4_train_eval')
importlib.reload(train_mod)

train_mod.seed_everything(CONFIG['seed'])
samples_meta = base_dataset.samples_meta
if len(samples_meta) < 6:
    raise RuntimeError(
        f"Only {len(samples_meta)} processed v3 MIL bags found in {PROCESSED_DIR}! "
        "Make sure Cell 2 downloaded raw MP4s into data/raw/ and Cell 4 finished preprocessing."
    )
n_folds = min(CONFIG['cv_folds'], max(2, len({m['game_id'] for m in samples_meta})))

fold_aucs, fold_f1s, fold_baccs, fold_type_accs = [], [], [], []
fold_models = []
oof_logits_list, oof_targets_list = [], []
oof_type_logits_list, oof_type_targets_list = [], []
best_fold_model = None
best_val_meta = []
best_score = -1.0

t0 = time.time()
for f_idx in range(n_folds):
    tr_m, va_m = model_mod.game_grouped_train_val_split(samples_meta, n_splits=n_folds, fold_idx=f_idx, seed=CONFIG['seed'])
    print(f"\n--- Fold [{f_idx+1}/{n_folds}] Train: {len(tr_m)} clips ({len({m['game_id'] for m in tr_m})} games) | Val: {len(va_m)} clips ({len({m['game_id'] for m in va_m})} unseen games) ---")
    f_model, f_metrics, f_vloss, f_oof = train_mod._train_single_fold(
        train_meta=tr_m,
        val_meta=va_m,
        epochs=CONFIG['epochs'],
        batch_size=CONFIG['batch_size'],
        lr=CONFIG['lr'],
        weight_decay=CONFIG['weight_decay'],
        dropout=CONFIG['dropout'],
        device=DEVICE,
        seed=CONFIG['seed'] + f_idx,
        return_oof=True,
    )
    fold_models.append(f_model)
    fold_aucs.append(f_metrics['roc_auc'])
    fold_f1s.append(f_metrics['f1'])
    fold_baccs.append(f_metrics['balanced_accuracy'])
    fold_type_accs.append(f_metrics['call_type_accuracy'])
    oof_logits_list.append(f_oof['logits'])
    oof_targets_list.append(f_oof['targets'])
    if len(f_oof['type_targets']) > 0:
        oof_type_logits_list.append(f_oof['type_logits'])
        oof_type_targets_list.append(f_oof['type_targets'])
    print(f"   ✅ Fold {f_idx+1} Out-of-Game Best -> ROC-AUC: {f_metrics['roc_auc']:.3f} | F1: {f_metrics['f1']:.3f} | BalAcc: {f_metrics['balanced_accuracy']:.3f} | CallType Acc: {f_metrics['call_type_accuracy']:.3f}")

    score = 0.55 * f_metrics['roc_auc'] + 0.45 * f_metrics['f1']
    if score > best_score:
        best_score = score
        best_fold_model = f_model
        best_val_meta = va_m

# Fit ONE Global Out-of-Fold (OOF) Temperature & Decision Threshold across ALL unseen-game clips!
all_oof_logits = np.concatenate(oof_logits_list, axis=0)
all_oof_targets = np.concatenate(oof_targets_list, axis=0)
oof_temp, oof_thresh, oof_metrics = train_mod.fit_temperature_and_threshold(all_oof_logits, all_oof_targets)
if oof_type_targets_list:
    all_tl = np.concatenate(oof_type_logits_list, axis=0)
    all_tt = np.concatenate(oof_type_targets_list, axis=0)
    oof_metrics['call_type_accuracy'] = float(((1.0 / (1.0 + np.exp(-all_tl))) >= 0.5).astype(int).__eq__(all_tt).mean())
else:
    oof_metrics['call_type_accuracy'] = float(np.mean(fold_type_accs))

best_fold_model.temperature.fill_(float(oof_temp))
model = model_mod.FoulMILEnsemble(fold_models, temperature=oof_temp).to(DEVICE)

ckpt_payload = {
    'architecture': 'FoulMILEnsemble_Swin3D_v3',
    'model_state_dict': {k: v.detach().cpu() for k, v in best_fold_model.state_dict().items()},
    'ensemble_state_dicts': [{k: v.detach().cpu() for k, v in m.state_dict().items()} for m in fold_models],
    'metrics': oof_metrics,
    'optimal_threshold': float(oof_thresh),
    'temperature': float(oof_temp),
    'cv_fold_summary': {
        'cv_folds': n_folds,
        'mean_roc_auc': float(np.mean(fold_aucs)),
        'std_roc_auc': float(np.std(fold_aucs)),
        'mean_f1': float(np.mean(fold_f1s)),
        'std_f1': float(np.std(fold_f1s)),
        'mean_call_type_acc': float(np.mean(fold_type_accs)),
        'oof_total_clips': int(len(all_oof_targets)),
    },
}
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(ckpt_payload, CHECKPOINT_PATH)
torch.save(ckpt_payload, PROJECT_DIR / 'best_foul_r2plus1d.pt')

print("\n" + "=" * 92)
print(f"🏁 {n_folds}-Fold Leak-Free Out-of-Game Ensemble Completed in {time.time()-t0:.1f}s!")
print(f"   • Mean Fold ROC-AUC           : {np.mean(fold_aucs):.3f} ± {np.std(fold_aucs):.3f}")
print(f"   • Mean Fold F1                : {np.mean(fold_f1s):.3f} ± {np.std(fold_f1s):.3f}")
print(f"   • Global OOF ROC-AUC ({len(all_oof_targets)} clips): {oof_metrics['roc_auc']:.3f} | Global OOF F1: {oof_metrics['f1']:.3f} | BalAcc: {oof_metrics['balanced_accuracy']:.3f}")
print(f"   • Global OOF Call-Type Acc    : {oof_metrics['call_type_accuracy']:.3f} (Shooting Foul vs. Personal Foul)")
print(f"   • Saved {n_folds}-Fold Ensemble Ckpt : {CHECKPOINT_PATH} (Global OOF T={oof_temp:.2f}, Thresh={oof_thresh:.2f})")
print("=" * 92)

fig, ax = plt.subplots(1, 1, figsize=(10, 4), dpi=100)
x = np.arange(1, n_folds + 1)
ax.bar(x - 0.25, fold_aucs, width=0.25, label='ROC-AUC', color='#d62728')
ax.bar(x, fold_f1s, width=0.25, label='F1 Score', color='#2ca02c')
ax.bar(x + 0.25, fold_type_accs, width=0.25, label='Call-Type Acc', color='#1f77b4')
ax.axhline(oof_metrics['roc_auc'], color='#d62728', linestyle='--', alpha=0.65, label=f"Global OOF ROC-AUC ({oof_metrics['roc_auc']:.3f})")
ax.set_xticks(x)
ax.set_xticklabels([f'Fold {i}' for i in x])
ax.set_ylim(0, 1.05)
ax.set_title('Leak-Free Out-of-Game 5-Fold Cross-Validation & Global OOF Ensemble (Grouped by NBA game_id)', fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)
ax.legend(ncol=2)
plt.tight_layout()
plt.show()

---
## Stage 5: Multi-Window Sequential RL Referee Agent (`NBARefereeEnv` + Double Dueling DQN)
Unlike static-vector RL toys, every timestep $k = 0 \dots K-1$ in v3.0 is a **distinct 2.0-second live-play window** carrying its own `Swin3D-T` visual embedding, its own calibrated MIL foul probability $p_{\text{MIL}}(k)$ and temporal delta $\Delta p_{\text{MIL}}(k)$, its own 16-D biomechanical summary, and **real NBA L2M review rewards** (`CC`, `INC`, `CNC`, `IC`).

In [ ]:
# ==============================================================================
# CELL 7: Train Multi-Window Sequential RL Referee Agent (Double Dueling DQN)
# ==============================================================================
val_ids = {m['play_id'] for m in best_val_meta}
train_ds_rl = model_mod.L2MFoulDataset(samples=[m for m in samples_meta if m['play_id'] not in val_ids], is_train=False)
val_ds_rl = model_mod.L2MFoulDataset(samples=best_val_meta, is_train=False)

train_episodes = model_mod.create_referee_episodes_from_dataset(train_ds_rl, mil_model=model, device=DEVICE)
val_episodes = model_mod.create_referee_episodes_from_dataset(val_ds_rl, mil_model=model, device=DEVICE)
print(f"🎮 Created {len(train_episodes)} train episodes & {len(val_episodes)} unseen-game val episodes (multi-window trajectories).")

rl_metrics = train_mod.train_rl_referee_agent(
    episodes=train_episodes,
    num_episodes=CONFIG['rl_episodes'],
    device=DEVICE,
    checkpoint_path=RL_CHECKPOINT_PATH,
)

# Evaluate trained RL Referee strictly on unseen-game validation episodes
rl_agent = model_mod.RefereeRLAgent(device=DEVICE)
rl_agent.load(RL_CHECKPOINT_PATH)
val_rl_correct = []
for ep in val_episodes:
    res = rl_agent.evaluate_episode(ep)
    val_rl_correct.append(int(res['is_foul']) == int(ep['label']))
print(f"🏆 Unseen-Game Validation RL Officiating Accuracy: {np.mean(val_rl_correct)*100:.1f}% ({sum(val_rl_correct)}/{len(val_rl_correct)})")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4), dpi=100)
rews = rl_metrics['episode_rewards']
ax1.plot(range(1, len(rews)+1), rews, alpha=0.3, color='#1f77b4', label='Episode Return')
if len(rews) >= 20:
    roll = np.convolve(rews, np.ones(20)/20, mode='valid')
    ax1.plot(range(20, len(rews)+1), roll, lw=2, color='#d62728', label='20-Ep Rolling Avg')
ax1.set_title('Multi-Window RL Referee Return (Real L2M Rewards)', fontweight='bold')
ax1.set_xlabel('Episode')
ax1.set_ylabel('Return')
ax1.grid(True, alpha=0.3)
ax1.legend()

if rl_metrics['losses']:
    ax2.plot(rl_metrics['losses'], color='#2ca02c')
    ax2.set_title('Double Dueling DQN Huber Loss', fontweight='bold')
    ax2.set_xlabel('Episode')
    ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Stage 6: Window-by-Window MIL Attention & RL Whistle Telemetry
Inspects an unseen-game validation play across all 2.0-second windows, showing how **MIL foul probability**, **RL $Q(\text{Whistle})$ vs. $Q(\text{Play On})$**, and **depth-gated Rule 12 biomechanics** evolve over time.

In [ ]:
# ==============================================================================
# CELL 8: Multi-Window MIL + RL Decision Telemetry on an Unseen-Game Play
# ==============================================================================
# Pick the first positive foul episode in unseen validation games (or index 0)
eval_ep = next((e for e in val_episodes if e['label'] == 1), val_episodes[0])
rl_res = rl_agent.evaluate_episode(eval_ep)

print(f"🎬 Unseen-Game Play : {eval_ep['play_id']} (Game {eval_ep['game_id']})")
print(f"   • Ground Truth   : {'FOUL' if eval_ep['label']==1 else 'NO FOUL'} ({eval_ep['call_type']} | {eval_ep['review_decision']})")
print(f"   • RL Decision    : {rl_res['decision']} (Step: {rl_res['whistle_step']} / {eval_ep['num_steps']}, Conf: {rl_res['confidence']*100:.1f}%)")

steps_x = range(1, eval_ep['num_steps'] + 1)
q_play = [q[0] for q in rl_res['q_values']]
q_foul = [q[1] for q in rl_res['q_values']]
p_sum = eval_ep['pose_summary']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4.5), dpi=100)
ax1.plot(steps_x, q_play, 'o-', color='#2ca02c', label='Q(Play On)', lw=2)
ax1.plot(steps_x, q_foul, 's-', color='#d62728', label='Q(Whistle Foul)', lw=2)
ax1.plot(steps_x, eval_ep['mil_probs'], '^--', color='#ff7f0e', label='Calibrated MIL P(Foul)', lw=2)
if rl_res['whistle_step'] is not None:
    ax1.axvline(rl_res['whistle_step'] + 1, color='red', linestyle=':', lw=2, label='RL Whistle Trigger')
ax1.set_title(f'Sequential Window Action-Values — {eval_ep["play_id"]}', fontweight='bold')
ax1.set_xlabel('2.0s Window Index Across Play')
ax1.grid(True, alpha=0.3)
ax1.legend()

ax2.plot(steps_x, p_sum[:, 0], 'o-', label='Arm Contact Intrusion', color='#e377c2')
ax2.plot(steps_x, p_sum[:, 1], 's-', label='Depth-Gated Cylinder Pen.', color='#ff7f0e')
ax2.plot(steps_x, p_sum[:, 12], '^-', label='E-BARD Hoop Proximity', color='#1f77b4')
ax2.plot(steps_x, p_sum[:, 14], 'd--', label='E-BARD Ball On-Play', color='#2ca02c')
ax2.set_title('Depth-Gated Rule 12 & Ball/Hoop Kinematics', fontweight='bold')
ax2.set_xlabel('2.0s Window Index Across Play')
ax2.set_ylim(-0.05, 1.05)
ax2.grid(True, alpha=0.3)
ax2.legend()
plt.tight_layout()
plt.show()

---
## Stage 7: Autonomous AI Referee Testing on ANY Raw NBA Game Footage (`data/test/` $\to$ `data/annotated/`)
To test the AI Referee on **any raw NBA broadcast video** (10s highlight, 20s L2M clip, or multi-minute game footage—even from games completely outside training!):
1. **Drop Raw Footage into `data/test/`**: Place any `.mp4`, `.mov`, `.avi`, `.mkv`, or `.webm` files into `NBA-Foul-Classifier-WIP/data/test/`.
2. **Run Cell 9**: The sliding-window Swin3D-T MIL AI Referee + E-BARD Zero-Referee Tracker runs autonomously (zero CSV metadata used!) and exports annotated replays to `data/annotated/<video_stem>_annotated.mp4`.
   - **When Foul(s) Occur**: Highlights **WHAT foul it is**, **WHERE it occurred** (opposing-team boxes only—zero referees!), and **WHEN it occurred** strictly during each foul window.
   - **When NO Foul Occurs**: **Nothing happens** on the output video (100% untouched playback).

In [ ]:
# ==============================================================================
# CELL 9: Drop-Folder Autonomous AI Referee Testing (Any-Length NBA Footage!)
# ==============================================================================
train_mod = importlib.import_module('4_train_eval')
importlib.reload(train_mod)

TEST_DIR.mkdir(parents=True, exist_ok=True)
ANNOTATED_DIR.mkdir(parents=True, exist_ok=True)

test_clips = train_mod.discover_video_files_in_folder(TEST_DIR)
print(f"📂 Raw Game Footage Folder : {TEST_DIR}")
print(f"🎬 Annotated Output Folder : {ANNOTATED_DIR}")
print(f"🔍 Discovered {len(test_clips)} raw video file(s):")
for i, c in enumerate(test_clips, 1):
    print(f"   [{i}] {c.name} ({c.stat().st_size / (1024*1024):.1f} MB)")

referee_reports = train_mod.run_ai_referee_on_folder(
    folder_path=TEST_DIR,
    checkpoint_path=CHECKPOINT_PATH,
    annotated_dir=ANNOTATED_DIR,
    yolo_weights=preprocess_mod.EBARD_WEIGHTS_NAME,
    pose_weights=str(WEIGHTS_DIR / 'yolo11n-pose.pt'),
)

if referee_reports:
    print("\n" + "=" * 124)
    print("🏀 AI REFEREE OFFICIATING REPORT — ALL CLIPS IN data/test/ (100% AUTONOMOUS, ZERO CSV METADATA)")
    print("=" * 124)
    print(f"{'VIDEO CLIP':<25} | {'CALL':<16} | {'WHAT FOUL':<40} | {'WHEN':<18} | {'WHERE'}")
    print("-" * 124)
    for r in referee_reports:
        badge = f"🚨 FOUL ({r['foul_prob']*100:.0f}%)" if r['is_foul'] else f"✅ NO FOUL ({r['foul_prob']*100:.0f}%)"
        what_s = r['foul_subtype'] if r['is_foul'] else "None (Nothing Happens)"
        when_s = f"{r['when_time_sec']:.2f}s (F{r['when_frame']})" if r['is_foul'] else "N/A"
        where_s = r['where_zone'] if r['is_foul'] else "N/A"
        print(f"{r['clip']:<25} | {badge:<16} | {what_s:<40} | {when_s:<18} | {where_s}")
    print("=" * 124)
else:
    print(f"⚠️ No video files found in {TEST_DIR}. Drop any raw NBA .mp4/.mov video into {TEST_DIR} and re-run.")

In [ ]:
# ==============================================================================
# CELL 10: Interactive Multi-Video HTML5 AI Referee Replay Console (WHAT, WHERE, WHEN)
# ==============================================================================
import base64
from IPython.display import HTML, display

raw_test_files = train_mod.discover_video_files_in_folder(TEST_DIR)
decisions_json = ANNOTATED_DIR / 'referee_decisions.json'
saved_decisions = {}
if decisions_json.exists():
    with open(decisions_json, 'r', encoding='utf-8') as jf:
        saved_decisions = json.load(jf)

annotated_videos = [
    ANNOTATED_DIR / f"{v.stem}_annotated.mp4"
    for v in raw_test_files
    if (ANNOTATED_DIR / f"{v.stem}_annotated.mp4").exists()
] if raw_test_files else sorted(ANNOTATED_DIR.glob('*_annotated.mp4'))

if annotated_videos:
    nav_tabs_html, video_cards_html = [], []
    for idx, vid_path in enumerate(annotated_videos):
        stem = vid_path.stem.replace('_annotated', '')
        dec = saved_decisions.get(stem, {})
        is_foul = bool(dec.get('is_foul', False))
        foul_prob = float(dec.get('foul_prob', 0.0))
        what_foul = dec.get('what_foul', 'No Foul')
        foul_subtype = dec.get('foul_subtype', 'No Foul (Clean Play)')
        foul_mechanic = dec.get('foul_mechanic', 'Rule 12B: Legal Guarding Position')
        when_str = dec.get('when_str', 'N/A (No Foul)')
        where_zone = dec.get('where_zone', 'N/A (No Foul)')

        with open(vid_path, 'rb') as vf:
            b64_vid = base64.b64encode(vf.read()).decode('utf-8')

        tab_style = "background: #00e5ff; color: #000; font-weight: bold;" if idx == 0 else "background: #2a2a2a; color: #ccc;"
        disp_style = "block" if idx == 0 else "none"
        status_color = "#ff1744" if is_foul else "#00c853"
        status_lbl = f"FOUL WHISTLED: {what_foul.upper()} ({foul_prob*100:.1f}%)" if is_foul else f"NO FOUL — PLAY ON ({foul_prob*100:.1f}%)"

        nav_tabs_html.append(f'''
        <button id="btn-clip-{idx}" onclick="showClip({idx}, {len(annotated_videos)})"
                style="padding: 8px 14px; margin: 4px; border: 1px solid #444; border-radius: 6px; cursor: pointer; font-size: 13px; {tab_style}">
            {"🚨" if is_foul else "✅"} Clip {idx+1}: {stem} ({what_foul})
        </button>''')

        if is_foul:
            banner = f'''
            <div style="display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 10px; margin-bottom: 12px; background: #221418; border: 1px solid #ff1744; border-radius: 6px; padding: 10px; font-size: 12.5px; color: #eee;">
                <div><strong style="color: #ff5252;">WHAT FOUL:</strong><br>{foul_subtype}<br><span style="color: #aaa; font-size: 11px;">{foul_mechanic}</span></div>
                <div><strong style="color: #00e5ff;">WHEN IT OCCURRED:</strong><br>{when_str}</div>
                <div><strong style="color: #ffd740;">WHERE IT OCCURRED:</strong><br>{where_zone}</div>
            </div>'''
        else:
            banner = '''
            <div style="margin-bottom: 12px; background: #132418; border: 1px solid #00c853; border-radius: 6px; padding: 10px; font-size: 12.5px; color: #b9f6ca;">
                <strong>✅ NO FOUL DETECTED:</strong> Legal play maintained. Nothing happens on the video (clean unannotated playback).
            </div>'''

        video_cards_html.append(f'''
        <div id="clip-card-{idx}" style="display: {disp_style}; margin-top: 12px; background: #181818; border: 1px solid #333; border-radius: 8px; padding: 14px;">
            <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 10px; border-bottom: 1px solid #282828; padding-bottom: 8px;">
                <div>
                    <span style="color: #fff; font-size: 16px; font-weight: bold;">🏀 AI Referee Call — {stem}</span>
                    <span style="margin-left: 10px; background: {status_color}; color: white; padding: 4px 10px; border-radius: 4px; font-weight: bold; font-size: 12px;">{status_lbl}</span>
                </div>
                <div style="color: #888; font-size: 12px;">Output: <code style="color: #00e5ff;">{vid_path.name}</code></div>
            </div>
            {banner}
            <video width="100%" controls autoplay loop muted style="border-radius: 6px; background: #000;">
                <source src="data:video/mp4;base64,{b64_vid}" type="video/mp4">
            </video>
        </div>''')

    html = f'''
    <div style="background: #121212; padding: 18px; border-radius: 10px; border: 2px solid #00e5ff; max-width: 960px; font-family: -apple-system, BlinkMacSystemFont, sans-serif;">
        <h3 style="color: #fff; margin: 0 0 8px 0;">🏀 NBA AI Referee (v3.0 E-BARD + Swin3D-T MIL) — Replay Console ({len(annotated_videos)} Clips)</h3>
        <p style="color: #aaa; font-size: 12px; margin: 0 0 12px 0;">Drop any raw NBA game video into <code>data/test/</code> and run Cell 9/10. Referees & spectators are explicitly filtered via E-BARD YOLO + 2-Team CIELAB clustering.</p>
        <div style="display: flex; flex-wrap: wrap; border-bottom: 1px solid #333; padding-bottom: 10px;">{"".join(nav_tabs_html)}</div>
        {"".join(video_cards_html)}
    </div>
    <script>
    function showClip(sel, n) {{
        for (let i = 0; i < n; i++) {{
            let c = document.getElementById("clip-card-" + i), b = document.getElementById("btn-clip-" + i);
            if (!c || !b) continue;
            let on = (i === sel);
            c.style.display = on ? "block" : "none";
            b.style.background = on ? "#00e5ff" : "#2a2a2a";
            b.style.color = on ? "#000" : "#ccc";
            b.style.fontWeight = on ? "bold" : "normal";
            let v = c.querySelector("video");
            if (v) {{ if (on) v.play(); else v.pause(); }}
        }}
    }}
    </script>'''
    display(HTML(html))
else:
    print(f"⚠️ No annotated replays found yet. Place raw videos in {TEST_DIR} and run Cell 9.")